In [3]:
from pathlib import Path

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
import requests

zip_file = DATA_DIR / "complaints.csv.zip"

if zip_file.exists():
    zip_file.unlink()

url = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

with requests.get(url, stream=True) as r:
    r.raise_for_status()

    with open(DATA_DIR / "complaints.csv.zip", "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print("Download complete")

Download complete


In [5]:
import zipfile

with zipfile.ZipFile(DATA_DIR / "complaints.csv.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted")

Extracted


In [6]:
import duckdb

duckdb.sql("""
COPY (
    SELECT *
    FROM read_csv_auto(
        '../data/raw/complaints.csv'
    )
)
TO '../data/raw/consumer_complaints.parquet'
(FORMAT PARQUET);
""")

### TODO
Add filter to the dataset to expand to the dataset to the other 4 categories

In [1]:
import polars as pl

products = [
    "Mortgage",
    "Checking or savings account",
    "Credit card",
    "Credit card or prepaid card",
    "Bank account or service"
]

response = [
    "Closed with monetary relief",
    "Closed with non-monetary relief"
]

(
    pl.scan_parquet("../data/raw/consumer_complaints.parquet")
    .filter(pl.col("Product").is_in(products))
    .filter(
        (pl.col("Date received") >= pl.date(2016, 1, 1))
        & (pl.col("Date received") < pl.date(2026, 5, 31))
    )
    .collect()
    .write_parquet(
        "../data/processed/consumer_banking_complaints.parquet"
    )
)

(
    pl.scan_parquet("../data/raw/consumer_complaints.parquet")
    .filter(pl.col("Product").is_in(products))
    .filter(
        (pl.col("Date received") >= pl.date(2016, 1, 1))
        & (pl.col("Date received") < pl.date(2026, 5, 31))
    )
    .filter(pl.col("Company response to consumer").is_in(response))
    .collect()
    .write_parquet(
        "../data/processed/consumer_banking_monetary_complaints.parquet"
    )
)

relief_response = [
    "Closed with monetary relief",
    "Closed with non-monetary relief",
    "Closed with explanation"
]

(
    pl.scan_parquet("../data/raw/consumer_complaints.parquet")
    .filter(pl.col("Product").is_in(products))
    .filter(
        (pl.col("Date received") >= pl.date(2016, 1, 1))
        & (pl.col("Date received") < pl.date(2026, 5, 31))
    )
    .filter(pl.col("Company response to consumer").is_in(relief_response))
    .collect()
    .write_parquet(
        "../data/processed/consumer_banking_relief.parquet"
    )
)
